# Batch Inference with VLM on OCR-VQA

This notebook runs batch inference using your imported HuggingFace model on book cover images and evaluates text reading accuracy.

In [ ]:
pip install -U ipywidgets

In [ ]:
!pip install --upgrade snowflake-ml-python>=1.8.0 -q

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import json

session = get_active_session()

DB_NAME = "VQA_DEMO_DB"
SCHEMA_NAME = "VQA"
MODEL_NAME = "LLAVA_V1_6_MISTRAL_7B_HF"  # Updated to match actual model name
MODEL_VERSION = "v1"

session.sql(f"USE DATABASE {DB_NAME}").collect()
session.sql(f"USE SCHEMA {SCHEMA_NAME}").collect()

reg = Registry(session=session, database_name=DB_NAME, schema_name=SCHEMA_NAME)

print(f"Connected to Snowflake")

## Load Model from Registry

In [ ]:
model = reg.get_model(MODEL_NAME)
mv = model.default

print(f"Model: {MODEL_NAME}")
print(f"Version: {mv._version_name}")
print(f"Functions: {[f['name'] for f in mv.show_functions()]}")

## Load Input Data

In [ ]:
input_table = session.table(f"{DB_NAME}.{SCHEMA_NAME}.INFERENCE_INPUT")
test_data = input_table.to_pandas()

print(f"Loaded {len(test_data)} samples for inference")
print(f"\nQuestion types:")
print(test_data['QUESTION_TYPE'].value_counts())

## Run Batch Inference

In [ ]:
from snowflake.ml.model.batch import JobSpec, OutputSpec, SaveMode, InputSpec
import time

OUTPUT_STAGE = f"@{DB_NAME}.{SCHEMA_NAME}.DATA_STAGE/inference_output/"
COMPUTE_POOL = "GPU_INFERENCE_POOL"

test_data_subset = test_data.head(10)
input_df = session.create_dataframe(test_data_subset[['MESSAGES']])

print(f"Starting batch inference on {input_df.count()} samples...")
print(f"Output: {OUTPUT_STAGE}")
start_time = time.time()

job = mv.run_batch(
    compute_pool=COMPUTE_POOL,
    X=input_df,
    input_spec=InputSpec(params={"temperature": 0.1, "max_completion_tokens": 100}),
    output_spec=OutputSpec(stage_location=OUTPUT_STAGE, mode=SaveMode.OVERWRITE),
    job_spec=JobSpec(gpu_requests="1")
)

print("Job submitted. Waiting for completion...")
job.wait()

elapsed = time.time() - start_time
print(f"\nCompleted in {elapsed:.1f}s ({elapsed/len(test_data_subset):.1f}s per sample)")

## Load and Parse Results

In [ ]:
results_df = session.read.option("pattern", ".*\\.parquet").parquet(OUTPUT_STAGE)
results_pd = results_df.to_pandas()

print(f"Loaded {len(results_pd)} results")
print(f"Columns: {results_pd.columns.tolist()}")

def extract_prediction(row):
    try:
        choices = row.get('id')
        if isinstance(choices, str):
            choices = json.loads(choices)
        if choices and len(choices) > 0:
            return choices[0]['message']['content'].strip()
    except:
        pass
    return None

results_pd['PREDICTION'] = results_pd.apply(extract_prediction, axis=1)

print("\nSample predictions:")
for i in range(min(3, len(results_pd))):
    print(f"  {i+1}: {results_pd.iloc[i]['PREDICTION'][:80]}...")

## Evaluate Results

In [ ]:
import re

def normalize(text):
    if not text:
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

def is_correct(predicted, expected, q_type):
    if not predicted:
        return False
    
    pred = normalize(predicted)
    exp = normalize(expected)
    
    if exp in pred:
        return True
    if pred.startswith(exp):
        return True
    for word in exp.split():
        if len(word) > 3 and word in pred:
            return True
    return False

eval_data = test_data_subset.copy()
eval_data['PREDICTED'] = results_pd['PREDICTION'].values
eval_data['IS_CORRECT'] = eval_data.apply(
    lambda r: is_correct(r['PREDICTED'], r['ANSWER'], r['QUESTION_TYPE']), axis=1
)

accuracy = eval_data['IS_CORRECT'].mean() * 100

print("=" * 50)
print("OCR-VQA EVALUATION RESULTS")
print("=" * 50)
print(f"Overall Accuracy: {accuracy:.1f}%")
print(f"Correct: {eval_data['IS_CORRECT'].sum()}/{len(eval_data)}")
print("=" * 50)

## View Detailed Results

In [ ]:
print("\nDetailed Results:")
print("-" * 80)

for _, row in eval_data.iterrows():
    status = "✓" if row['IS_CORRECT'] else "✗"
    print(f"\n{status} Q: {row['QUESTION']}")
    print(f"   Expected: {row['ANSWER']}")
    print(f"   Got: {str(row['PREDICTED'])[:100]}")

## Save Results

In [ ]:
results_to_save = eval_data[['ID', 'QUESTION', 'ANSWER', 'PREDICTED', 'QUESTION_TYPE', 'IS_CORRECT']].copy()
results_to_save.columns = ['ID', 'QUESTION', 'EXPECTED_ANSWER', 'PREDICTED_ANSWER', 'QUESTION_TYPE', 'IS_CORRECT']

session.create_dataframe(results_to_save).write.mode("overwrite").save_as_table(
    f"{DB_NAME}.{SCHEMA_NAME}.INFERENCE_RESULTS"
)

print(f"Results saved to {DB_NAME}.{SCHEMA_NAME}.INFERENCE_RESULTS")